# Census and HUD USER ingress test

This notebook downloads one small public sample file from each source and saves the response bytes unchanged. It does not parse, clean, reshape, or otherwise transform the data.

Sample files:

- Census: 2025 Alabama county population estimates by selected age groups and sex (CSV)
- HUD USER: FY2027 county-level Fair Market Rents (XLSX)

In [ ]:
# Run once per environment; pip skips the package if it is already satisfied.
%pip install -q requests

import hashlib
from pathlib import Path
from urllib.parse import unquote, urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configure sample files

In [ ]:
SAMPLE_FILES = [
    {
        "source": "census",
        "url": (
            "https://www2.census.gov/programs-surveys/popest/datasets/"
            "2020-2025/counties/asrh/cc-est2025-agesex-01.csv"
        ),
    },
    {
        "source": "huduser",
        "url": (
            "https://www.huduser.gov/portal/datasets/fmr/fmr2027/"
            "FY27_FMRs.xlsx"
        ),
    },
]

OUTPUT_DIR = Path("data/ingress_samples")
TIMEOUT_SECONDS = 60
ALLOWED_DOMAINS = ("census.gov", "huduser.gov")

## Download raw files

The helper validates both the requested URL and the final URL after redirects, retries temporary HTTP failures, and writes the response body directly to disk.

In [ ]:
def is_allowed_host(hostname):
    if not hostname:
        return False
    hostname = hostname.lower().rstrip(".")
    return any(
        hostname == domain or hostname.endswith(f".{domain}")
        for domain in ALLOWED_DOMAINS
    )


def download_raw_file(session, source, url):
    requested_url = urlparse(url)
    if requested_url.scheme != "https" or not is_allowed_host(requested_url.hostname):
        raise ValueError(f"Disallowed URL: {url}")

    response = session.get(url, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()

    final_url = urlparse(response.url)
    if final_url.scheme != "https" or not is_allowed_host(final_url.hostname):
        raise ValueError(f"Redirected to a disallowed URL: {response.url}")

    filename = Path(unquote(final_url.path)).name
    if not filename:
        raise ValueError(f"The final URL does not contain a filename: {response.url}")

    source_dir = OUTPUT_DIR / source
    source_dir.mkdir(parents=True, exist_ok=True)
    output_path = source_dir / filename
    output_path.write_bytes(response.content)

    return {
        "source": source,
        "status": response.status_code,
        "content_type": response.headers.get("content-type", "unknown"),
        "bytes": len(response.content),
        "sha256": hashlib.sha256(response.content).hexdigest(),
        "path": str(output_path.resolve()),
        "final_url": response.url,
    }

In [ ]:
retry_policy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}),
    respect_retry_after_header=True,
)
headers = {"User-Agent": "rental-analysis-ingress-test/1.0"}

results = []
with requests.Session() as session:
    session.headers.update(headers)
    session.mount("https://", HTTPAdapter(max_retries=retry_policy))

    for sample in SAMPLE_FILES:
        result = download_raw_file(session=session, **sample)
        results.append(result)
        print(
            f"[{result['source']}] HTTP {result['status']} | "
            f"{result['bytes']:,} bytes | {result['path']}"
        )

## Inspect ingress metadata

This displays request and file metadata only; the downloaded contents remain untouched.

In [ ]:
results